In [2]:
!pip install geopandas pyogrio



### Cell 1: Imports and config

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

# Paths to existing cleaned files
DC_PATH = "data_centers_cleaned.csv"
PLANTS_PATH = "plants_cleaned.csv"
PRICES_PATH = "electricity_price_cleaned.csv"

# Output
OUTPUT_PATH = "county_features.csv"

### Cell 2: Load Census county shapefile 

In [ ]:
# Census 2023 county boundaries, low-res 20m version
COUNTIES_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_20m.zip"

counties = gpd.read_file(COUNTIES_URL)

# Keep only continental US
EXCLUDE_STATEFP = {"02", "15", "60", "66", "69", "72", "78"}
counties = counties[~counties["STATEFP"].isin(EXCLUDE_STATEFP)].copy()

# Standardize FIPS as zero-padded 5-char string
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)

# Reproject to a US-friendly equal-area CRS for accurate distance calculations later
counties = counties.to_crs("EPSG:5070")

print(f"Loaded {len(counties)} continental US counties")
counties[["GEOID", "NAME", "STATEFP", "ALAND"]].head()

Loaded 3109 continental US counties


,GEOID,NAME,STATEFP,ALAND
0,13027,Brooks,13,1277341276
1,31095,Jefferson,31,1476765697
2,51683,Manassas,51,25493247
3,56021,Laramie,56,6956355036
4,13135,Gwinnett,13,1116606767


### Cell 3: Build a state FIPS → state name lookup 

In [5]:
# Census state FIPS codes for the 48 continental + DC
STATE_FIPS_TO_NAME = {
    "01": "Alabama", "04": "Arizona", "05": "Arkansas", "06": "California",
    "08": "Colorado", "09": "Connecticut", "10": "Delaware", "11": "District of Columbia",
    "12": "Florida", "13": "Georgia", "16": "Idaho", "17": "Illinois",
    "18": "Indiana", "19": "Iowa", "20": "Kansas", "21": "Kentucky",
    "22": "Louisiana", "23": "Maine", "24": "Maryland", "25": "Massachusetts",
    "26": "Michigan", "27": "Minnesota", "28": "Mississippi", "29": "Missouri",
    "30": "Montana", "31": "Nebraska", "32": "Nevada", "33": "New Hampshire",
    "34": "New Jersey", "35": "New Mexico", "36": "New York", "37": "North Carolina",
    "38": "North Dakota", "39": "Ohio", "40": "Oklahoma", "41": "Oregon",
    "42": "Pennsylvania", "44": "Rhode Island", "45": "South Carolina",
    "46": "South Dakota", "47": "Tennessee", "48": "Texas", "49": "Utah",
    "50": "Vermont", "51": "Virginia", "53": "Washington", "54": "West Virginia",
    "55": "Wisconsin", "56": "Wyoming"
}

counties["state_name"] = counties["STATEFP"].map(STATE_FIPS_TO_NAME)
print(f"States represented: {counties['state_name'].nunique()}")

States represented: 49


### Cell 4: Spatial join data centers → counties


In [ ]:
dc = pd.read_csv(DC_PATH)
print(f"Loaded {len(dc)} data centers. Columns: {list(dc.columns)}")

# Build GeoDataFrame from lat/lon
dc_gdf = gpd.GeoDataFrame(
    dc,
    geometry=gpd.points_from_xy(dc["lon"], dc["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:5070") 

# Spatial join: which county does each data center fall in?
dc_joined = gpd.sjoin(dc_gdf, counties[["GEOID", "geometry"]], how="left", predicate="within")
unmatched = dc_joined["GEOID"].isna().sum()
print(f"Unmatched data centers: {unmatched} ({unmatched/len(dc_joined)*100:.1f}%)")

# Aggregate per county: count and total sqft
dc_per_county = dc_joined.groupby("GEOID").agg(
    dc_count=("id", "count"),
    dc_total_sqft=("sqft", "sum")
).reset_index()

print(f"Counties with at least one data center: {len(dc_per_county)}")
dc_per_county.head()

Loaded 1472 data centers. Columns: ['id', 'state', 'state_abb', 'county', 'county_id', 'lat', 'lon', 'type', 'sqft']
Unmatched data centers: 1 (0.1%)
Counties with at least one data center: 247


,GEOID,dc_count,dc_total_sqft
0,01069,1,10736.0
1,01071,3,7869085.0
2,01073,1,665569.0
3,01089,6,16255032.0
4,04003,1,7539.0


### Cell 5: Spatial join renewable plants → counties


In [7]:
plants = pd.read_csv(PLANTS_PATH)
print(f"Loaded {len(plants)} plants. Columns: {list(plants.columns)}")

plants_gdf = gpd.GeoDataFrame(
    plants,
    geometry=gpd.points_from_xy(plants["Longitude"], plants["Latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:5070")

plants_joined = gpd.sjoin(plants_gdf, counties[["GEOID", "geometry"]], how="left", predicate="within")
unmatched_p = plants_joined["GEOID"].isna().sum()
print(f"Unmatched plants: {unmatched_p}")

# Aggregate: total in-county renewable capacity
renew_per_county = plants_joined.groupby("GEOID").agg(
    plant_count=("Plant Code", "count"),
    renewable_mw_in_county=("Nameplate Capacity (MW)", "sum")
).reset_index()

renew_per_county.head()

Loaded 9294 plants. Columns: ['Plant Code', 'Plant Name', 'State', 'Latitude', 'Longitude', 'Energy Source', 'Nameplate Capacity (MW)']
Unmatched plants: 101


,GEOID,plant_count,renewable_mw_in_county
0,01015,1,7.4
1,01017,1,79.2
2,01019,1,29.2
3,01021,1,29.5
4,01033,1,227.0


### Cell 6: Renewable capacity within 50km of county centroid 

In [ ]:
# County centroids in meters (EPSG:5070)
county_centroids = counties.copy()
county_centroids["centroid"] = county_centroids.geometry.centroid
centroid_coords = np.array([(p.x, p.y) for p in county_centroids["centroid"]])

# Plant coordinates in meters
plant_coords = np.array([(p.x, p.y) for p in plants_gdf.geometry])
plant_capacity = plants_gdf["Nameplate Capacity (MW)"].fillna(0).values

# Build KDTree on plants and query 50km neighbors per county centroid
tree = cKDTree(plant_coords)
RADIUS_M = 50_000  # 50 km in meters

renewable_within_50km = []
for i, c in enumerate(centroid_coords):
    idx = tree.query_ball_point(c, r=RADIUS_M)
    renewable_within_50km.append(plant_capacity[idx].sum())

county_centroids["renewable_mw_within_50km"] = renewable_within_50km

# Table for merging
proximity_df = county_centroids[["GEOID", "renewable_mw_within_50km"]]
proximity_df.head()

,GEOID,renewable_mw_within_50km
0,13027,379.7
1,31095,818.2
2,51683,124.8
3,56021,503.6
4,13135,108.5


### Cell 7: Add electricity price (state-level → broadcast to counties)


In [9]:
prices = pd.read_csv(PRICES_PATH)
print(f"Prices: {list(prices.columns)}")

# prices has columns: state, electricity_price
# Join via state name
prices_to_merge = prices.rename(columns={"state": "state_name"})
counties_with_price = counties.merge(prices_to_merge, on="state_name", how="left")

missing_price = counties_with_price["electricity_price"].isna().sum()
print(f"Counties missing price: {missing_price}")

Prices: ['state', 'electricity_price']
Counties missing price: 1


### Cell 8: Assemble the master county feature table


In [ ]:
# Start from counties, left-join
features = counties_with_price[["GEOID", "NAME", "STATEFP", "state_name", "ALAND"]].copy()
features = features.rename(columns={"NAME": "county_name", "ALAND": "land_area_m2"})

# Merge electricity price back in
features = features.merge(
    counties_with_price[["GEOID", "electricity_price"]], on="GEOID", how="left"
)

# Data center variables
features = features.merge(dc_per_county, on="GEOID", how="left")
features["dc_count"] = features["dc_count"].fillna(0).astype(int)
features["dc_total_sqft"] = features["dc_total_sqft"].fillna(0)
features["has_dc"] = (features["dc_count"] > 0).astype(int)  # binary target for ML

# Renewable in-county
features = features.merge(renew_per_county, on="GEOID", how="left")
features["plant_count"] = features["plant_count"].fillna(0).astype(int)
features["renewable_mw_in_county"] = features["renewable_mw_in_county"].fillna(0)

# Renewable within 50km
features = features.merge(proximity_df, on="GEOID", how="left")
features["renewable_mw_within_50km"] = features["renewable_mw_within_50km"].fillna(0)

# Land area in km², population density placeholder
features["land_area_km2"] = features["land_area_m2"] / 1e6

print(features.shape)
features.head()

(3109, 13)


,GEOID,county_name,STATEFP,state_name,land_area_m2,electricity_price,dc_count,dc_total_sqft,has_dc,plant_count,renewable_mw_in_county,renewable_mw_within_50km,land_area_km2
0,13027,Brooks,13,Georgia,1277341276,7.53,0,0.0,0,2,300.0,379.7,1277.341276
1,31095,Jefferson,31,Nebraska,1476765697,7.35,0,0.0,0,1,74.8,818.2,1476.765697
2,51683,Manassas,51,Virginia,25493247,9.35,5,881180.0,1,0,0.0,124.8,25.493247
3,56021,Laramie,56,Wyoming,6956355036,8.30,18,21738866.0,1,6,503.6,503.6,6956.355036
4,13135,Gwinnett,13,Georgia,1116606767,7.53,4,2155213.0,1,0,0.0,108.5,1116.606767


In [11]:
features.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(features)} county rows to {OUTPUT_PATH}")
print(f"Counties with data centers: {features['has_dc'].sum()}")
print(f"Mean dc_count: {features['dc_count'].mean():.2f}")

Saved 3109 county rows to county_features.csv
Counties with data centers: 247
Mean dc_count: 0.47


### Section A: Census county population + density

In [ ]:
# Census county population estimates (Vintage 2024)
# This file has columns including STATE (2-digit FIPS), COUNTY (3-digit FIPS), 
# POPESTIMATE2024, and CTYNAME

POP_URL = "https://www2.census.gov/programs-surveys/popest/datasets/2020-2024/counties/totals/co-est2024-alldata.csv"

pop = pd.read_csv(POP_URL, encoding="latin-1")
print(f"Loaded {len(pop)} rows. Columns: {list(pop.columns)[:10]}...")
pop = pop[pop["COUNTY"] != 0].copy()

# Build 5-digit FIPS by zero-padding STATE (2) + COUNTY (3)
pop["GEOID"] = (
    pop["STATE"].astype(str).str.zfill(2) + pop["COUNTY"].astype(str).str.zfill(3)
)
pop_slim = pop[["GEOID", "POPESTIMATE2024"]].rename(
    columns={"POPESTIMATE2024": "population"}
)

print(f"County population rows: {len(pop_slim)}")
pop_slim.head()

Loaded 3195 rows. Columns: ['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'ESTIMATESBASE2020', 'POPESTIMATE2020', 'POPESTIMATE2021']...
County population rows: 3144


,GEOID,population
1,01001,61464
2,01003,261608
3,01005,24358
4,01007,22258
5,01009,60163


In [13]:
# Merge population into features and compute density
features = features.merge(pop_slim, on="GEOID", how="left")

missing_pop = features["population"].isna().sum()
print(f"Counties missing population: {missing_pop}")

# Population density: people per km² 
features["pop_density"] = features["population"] / features["land_area_km2"]

print(features[["county_name", "state_name", "population", "pop_density"]].head())

Counties missing population: 0
  county_name state_name  population  pop_density
0      Brooks    Georgia       16344    12.795328
1   Jefferson   Nebraska        7136     4.832182
2    Manassas   Virginia       43616  1710.884455
3     Laramie    Wyoming      101783    14.631657
4    Gwinnett    Georgia     1003869   899.035390


### Section B: CDC Social Vulnerability Index (county level)


In [ ]:
# CDC SVI 2022, county level
SVI_PATH = "initial data/SVI_2022_US_county.csv"

svi = pd.read_csv(SVI_PATH)
print(f"SVI columns: {list(svi.columns)[:15]}")  # first 15 to keep output short
print(f"Total SVI rows: {len(svi)}")

# Key column: FIPS (county FIPS) and RPL_THEMES (overall vulnerability percentile, 0-1)
# CDC uses -999 to indicate missing, replace with NaN
svi["RPL_THEMES"] = svi["RPL_THEMES"].replace(-999, np.nan)

# Standardize FIPS to 5-char string
svi["GEOID"] = svi["FIPS"].astype(str).str.zfill(5)

svi_slim = svi[["GEOID", "RPL_THEMES"]].rename(columns={"RPL_THEMES": "svi_score"})
svi_slim.head()

SVI columns: ['ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150']
Total SVI rows: 3144


,GEOID,svi_score
0,01001,0.2663
1,01003,0.3487
2,01005,0.9927
3,01007,0.8451
4,01009,0.6166


In [15]:
# Merge SVI into features
features = features.merge(svi_slim, on="GEOID", how="left")

missing_svi = features["svi_score"].isna().sum()
print(f"Counties missing SVI: {missing_svi}")
print(features[["county_name", "state_name", "svi_score"]].head())

Counties missing SVI: 0
  county_name state_name  svi_score
0      Brooks    Georgia     0.9275
1   Jefferson   Nebraska     0.2736
2    Manassas   Virginia     0.4884
3     Laramie    Wyoming     0.2514
4    Gwinnett    Georgia     0.6433


### Section C: WRI Aqueduct water stress (state level → broadcast to counties)


In [ ]:
AQUEDUCT_PATH = "initial data/Aqueduct40_rankings_download_Y2023M07D05.xlsx"

# Read the province_baseline sheet (state-level for the US)
aq = pd.read_excel(AQUEDUCT_PATH, sheet_name="province_baseline")
print(f"Aqueduct columns ({len(aq.columns)}):")
print(list(aq.columns))
print(f"\nTotal rows: {len(aq)}")
aq.head(3)

Aqueduct columns (12):
['gid_0', 'gid_1', 'name_0', 'name_1', 'indicator_name', 'weight', 'score', 'score_ranked', 'cat', 'label', 'un_region', 'wb_region']

Total rows: 42771


,gid_0,gid_1,name_0,name_1,indicator_name,weight,score,score_ranked,cat,label,un_region,wb_region
0,AFG,AFG.2_1,Afghanistan,Badghis,bws,Ind,5.0,92.5,4.0,Extremely High (>80%),Asia,South Asia
1,AFG,AFG.8_1,Afghanistan,Faryab,bws,Ind,5.0,92.5,4.0,Extremely High (>80%),Asia,South Asia
2,ARE,ARE.1_1,United Arab Emirates,Abu Dhabi,bws,Dom,5.0,8.0,4.0,Extremely High (>80%),Asia,Middle East & North Africa


In [17]:
# Filter to: United States, baseline water stress (bws), industrial weighting
# "Ind" weighting is most relevant since data centers are industrial water users
aq_us = aq[
    (aq["name_0"] == "United States") &
    (aq["indicator_name"] == "bws") &
    (aq["weight"] == "Ind")
].copy()

print(f"US state rows after filtering: {len(aq_us)}")
print(aq_us[["name_1", "score", "label"]].head(10))

US state rows after filtering: 51
               name_1     score                  label
2210          Arizona  4.423375  Extremely High (>80%)
3128       New Mexico  4.129831  Extremely High (>80%)
3811         Colorado  4.034449  Extremely High (>80%)
4580       California  3.919994          High (40-80%)
6590          Wyoming  3.551411          High (40-80%)
8232         Delaware  3.361336          High (40-80%)
8347            Idaho  3.348895          High (40-80%)
9985   North Carolina  3.169009          High (40-80%)
11018         Florida  3.061819          High (40-80%)
11477        Nebraska  3.023475          High (40-80%)


In [ ]:
# Slim table for merging
aq_slim = aq_us[["name_1", "score"]].rename(
    columns={"name_1": "state_name", "score": "water_stress_score"}
)

# Average duplicated states
aq_slim = aq_slim.groupby("state_name", as_index=False)["water_stress_score"].mean()

print(f"Unique US states in Aqueduct: {len(aq_slim)}")
print(aq_slim.sort_values("water_stress_score", ascending=False).head(10))

Unique US states in Aqueduct: 51
        state_name  water_stress_score
2          Arizona            4.423375
31      New Mexico            4.129831
5         Colorado            4.034449
4       California            3.919994
50         Wyoming            3.551411
7         Delaware            3.361336
12           Idaho            3.348895
33  North Carolina            3.169009
9          Florida            3.061819
27        Nebraska            3.023475


In [19]:
# Broadcast state-level water stress to all counties in that state
features = features.merge(aq_slim, on="state_name", how="left")

missing_ws = features["water_stress_score"].isna().sum()
print(f"Counties missing water stress: {missing_ws}")
print(features[["county_name", "state_name", "water_stress_score"]].head())

Counties missing water stress: 0
  county_name state_name  water_stress_score
0      Brooks    Georgia            2.213066
1   Jefferson   Nebraska            3.023475
2    Manassas   Virginia            2.571924
3     Laramie    Wyoming            3.551411
4    Gwinnett    Georgia            2.213066


In [20]:
print("Final feature columns:", list(features.columns))
print(f"Shape: {features.shape}")
print(f"Counties with data centers: {features['has_dc'].sum()}")
print("\nMissing value counts per column:")
print(features.isna().sum())

features.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved enriched table to {OUTPUT_PATH}")

Final feature columns: ['GEOID', 'county_name', 'STATEFP', 'state_name', 'land_area_m2', 'electricity_price', 'dc_count', 'dc_total_sqft', 'has_dc', 'plant_count', 'renewable_mw_in_county', 'renewable_mw_within_50km', 'land_area_km2', 'population', 'pop_density', 'svi_score', 'water_stress_score']
Shape: (3109, 17)
Counties with data centers: 247

Missing value counts per column:
GEOID                       0
county_name                 0
STATEFP                     0
state_name                  0
land_area_m2                0
electricity_price           1
dc_count                    0
dc_total_sqft               0
has_dc                      0
plant_count                 0
renewable_mw_in_county      0
renewable_mw_within_50km    0
land_area_km2               0
population                  0
pop_density                 0
svi_score                   0
water_stress_score          0
dtype: int64

Saved enriched table to county_features.csv


In [21]:
# Find the missing one
missing = features[features["electricity_price"].isna()]
print(missing[["county_name", "state_name"]])

              county_name            state_name
395  District of Columbia  District of Columbia


In [22]:
# Maryland's price as a reasonable proxy 
features["electricity_price"] = features["electricity_price"].fillna(
    features["electricity_price"].median()  # safe fallback
)

In [23]:
features.to_csv("county_features.csv", index=False)
print("Saved. Missing electricity_price:", features["electricity_price"].isna().sum())

Saved. Missing electricity_price: 0
